# 03 · Modelling — Silver → Gold
### Proyecto ETL FIFA 21 · Jorge Amat · David Plaza

**Requisito previo:** haber ejecutado `transformaciones.ipynb` (existe `fifa_catalog.silver.df_silver`).

**Qué hace este notebook:**

Construye un modelo analítico en **esquema en estrella** a partir de Silver, con granularidad de 1 fila por jugador:

- `dim_player` — atributos del jugador
- `dim_nationality` — catálogo de nacionalidades
- `dim_club` — catálogo de clubes
- `dim_position` — catálogo de posiciones
- `fact_player_stats` — tabla de hechos con las métricas numéricas, enlazada a las dimensiones por FK

Las 5 tablas se guardan en `fifa_catalog.gold` en modo `overwrite`.

**Este es el último notebook del pipeline.**

In [0]:
from pyspark.sql import functions as f
from pyspark.sql.functions import monotonically_increasing_id,explode,split,trim


In [0]:
df_silver = spark.table("fifa_catalog.silver.df_silver")
display(df_silver.limit(20))

longname,playerurl,nationality,age,overall_rating,potential,id,height,weight,foot,best_overall,best_position,growth,joined,value,wage,release_clause,attacking,crossing,finishing,heading_accuracy,short_passing,volleys,skill,dribbling,curve,fk_accuracy,long_passing,ball_control,movement,acceleration,sprint_speed,agility,reactions,balance,power,shot_power,jumping,stamina,strength,long_shots,mentality,aggression,interceptions,positioning,vision,penalties,composure,defending,marking,standing_tackle,sliding_tackle,goalkeeping,gk_diving,gk_handling,gk_kicking,gk_positioning,gk_reflexes,total_stats,base_stats,weak_foot_starss,skills_stars,atack_contribution,defense_contribution,international_reputation,pac,sho,pas,dri,def,phy,hits,team_name,contract,team_original,is_free,contract_raw,num_positions,contract_start_year,years_at_team_2021,value_player,age_category,offensive_score,defensive_score,two_footed,clutch_score,clutch_level,source_layer,processed_timestamp
Kevin De Bruyne,http://sofifa.com/player/192985/kevin-de-bruyne/210005/,Belgium,29,91,91,192985,180.3,69.9,Right,91,CAM,0,"Aug 30, 2015",87000000,370000,161000000,407,94,82,55,94,82,441,88,85,83,93,92,398,77,76,78,91,76,408,91,63,89,74,91,408,76,66,88,94,84,91,186,68,65,53,56,0,0,0,0,0,2304,485,5,4,High,High,4,76,86,93,88,64,78,163,Manchester City,2015 - 2023,Manchester City2015 ~ 2023,false,2015 - 2023,2,2015,6,Premium,Veteran,85.83,64.4,1,92.0,Elite,bronze,2026-09-07T19:08:30.240Z
Harry Kane,http://sofifa.com/player/202126/harry-kane/210005/,England,26,88,89,202126,188.0,88.9,Right,88,ST,1,"Jul 1, 2010",71000000,220000,140200000,420,75,94,85,81,85,395,80,80,68,83,84,367,66,69,69,90,73,424,91,79,84,84,86,382,81,35,93,83,90,91,130,56,36,38,54,0,0,0,0,0,2172,449,4,3,High,High,3,68,91,80,80,47,83,229,Tottenham Hotspur,2010 - 2024,Tottenham Hotspur2010 ~ 2024,false,2010 - 2024,1,2010,11,Premium,Prime,81.67,48.0,1,88.0,Elite,bronze,2026-09-07T19:08:30.240Z
Antoine Griezmann,http://sofifa.com/player/194765/antoine-griezmann/210005/,France,29,87,87,194765,175.3,73.0,Left,87,ST,0,"Jul 12, 2019",50500000,290000,103500000,425,83,88,83,84,87,429,87,86,85,82,89,425,80,79,91,92,83,402,81,90,86,63,82,382,73,49,89,85,86,89,162,59,54,49,63,0,0,0,0,0,2288,465,3,4,Medium,Medium,4,79,85,84,88,57,72,209,FC Barcelona,2019 - 2024,FC Barcelona2019 ~ 2024,false,2019 - 2024,3,2019,2,High,Veteran,85.33,54.4,0,88.67,Elite,bronze,2026-09-07T19:08:30.240Z
Riyad Mahrez,http://sofifa.com/player/204485/riyad-mahrez/210005/,Algeria,29,85,85,204485,177.8,67.1,Left,85,RW,0,"Jul 10, 2018",37500000,210000,69400000,366,83,79,48,80,76,418,90,84,78,75,91,427,87,79,92,81,88,350,79,60,76,55,80,321,48,39,80,84,70,84,98,45,31,22,54,0,0,0,0,0,2034,430,4,5,Medium,Medium,3,83,79,81,90,38,59,109,Manchester City,2018 - 2023,Manchester City2018 ~ 2023,false,2018 - 2023,2,2018,3,High,Veteran,85.17,37.0,1,83.0,Strong,bronze,2026-09-07T19:08:30.240Z
Toby Alderweireld,http://sofifa.com/player/184087/toby-alderweireld/210005/,Belgium,31,85,85,184087,185.4,81.2,Right,85,CB,0,"Jul 8, 2015",28500000,130000,54200000,308,64,45,82,79,38,351,62,63,69,82,75,331,60,65,59,85,62,375,78,82,78,79,58,338,81,85,52,62,58,86,262,88,89,85,66,0,0,0,0,0,2031,423,3,2,Medium,Medium,3,63,55,72,67,87,79,65,Tottenham Hotspur,2015 - 2023,Tottenham Hotspur2015 ~ 2023,false,2015 - 2023,1,2015,6,High,Veteran,60.5,85.0,0,77.67,Strong,bronze,2026-09-07T19:08:30.240Z
Thiago Emiliano da Silva,http://sofifa.com/player/164240/thiago-emiliano-da-silva/210005/,Brazil,35,85,85,164240,182.9,78.9,Right,85,CB,0,"Aug 28, 2020",11500000,93000,21900000,322,60,40,81,80,61,349,67,62,61,80,79,332,57,61,67,81,66,379,71,90,71,82,65,353,76,88,59,70,60,86,257,87,86,84,45,0,0,0,0,0,2037,420,3,2,Medium,High,4,59,54,72,71,86,78,84,Chelsea,2020 - 2021,Chelsea2020 ~ 2021,false,2020 - 2021,1,2020,1,Mid,Late,61.67,85.2,0,79.0,Strong,bronze,2026-09-07T19:08:30.240Z
Arthur Henrique Ramos Oliveira Melo,http://sofifa.com/player/230658/arthur-henrique-ramos-oliveira-melo/210005/,Brazil,23,84,

# DIMENSIÓN JUGADOR

In [0]:
dim_player = (
    df_silver
    .select(
        df_silver.id.alias("player_id"),
        "longname",
        "playerurl",
        "foot",
        "best_position",
        "num_positions",
        "weak_foot_starss",
        "skills_stars",
        "international_reputation",
        "two_footed",
        "age_category",
        "contract",
        "joined"
    )
    # si contract tiene año -> ese
    # si no, y joined existe -> año de joined
    # si tampoco existe -> 2021 (
    .withColumn(
        "contract_start_year",
        f.when(
            f.col("contract").rlike(r"(19\d{2}|20\d{2})"),
            f.regexp_extract(f.col("contract"), r"(19\d{2}|20\d{2})", 1).cast("int")
        ).when(
            f.col("joined").isNotNull(),
            f.year(f.to_date(f.col("joined"), "MMM d, yyyy")).cast("int")
        ).otherwise(f.lit(2021).cast("int"))
    )
    .withColumn(
        "years_at_team_2021",
        f.greatest(f.lit(0), f.lit(2021) - f.col("contract_start_year")).cast("int")
    )
    .dropDuplicates(["player_id"])
)

display(dim_player.limit(20))
dim_player.write.mode("overwrite").saveAsTable(
    "fifa_catalog.gold.dim_player"
)


player_id,longname,playerurl,foot,best_position,num_positions,weak_foot_starss,skills_stars,international_reputation,two_footed,age_category,contract,joined,contract_start_year,years_at_team_2021
192985,Kevin De Bruyne,http://sofifa.com/player/192985/kevin-de-bruyne/210005/,Right,CAM,2,5,4,4,1,Veteran,2015 - 2023,"Aug 30, 2015",2015,6
202126,Harry Kane,http://sofifa.com/player/202126/harry-kane/210005/,Right,ST,1,4,3,3,1,Prime,2010 - 2024,"Jul 1, 2010",2010,11
194765,Antoine Griezmann,http://sofifa.com/player/194765/antoine-griezmann/210005/,Left,ST,3,3,4,4,0,Veteran,2019 - 2024,"Jul 12, 2019",2019,2
204485,Riyad Mahrez,http://sofifa.com/player/204485/riyad-mahrez/210005/,Left,RW,2,4,5,3,1,Veteran,2018 - 2023,"Jul 10, 2018",2018,3
184087,Toby Alderweireld,http://sofifa.com/player/184087/toby-alderweireld/210005/,Right,CB,1,3,2,3,0,Veteran,2015 - 2023,"Jul 8, 2015",2015,6
164240,Thiago Emiliano da Silva,http://sofifa.com/player/164240/thiago-emiliano-da-silva/210005/,Right,CB,1,3,2,4,0,Late,2020 - 2021,"Aug 28, 2020",2020,1
230658,Arthur Henrique Ramos Oliveira Melo,http://sofifa.com/player/230658/arthur-henrique-ramos-oliveira-melo/210005/,Right,CM,1,3,4,2,0,Prime,2020 - 2024,"Sep 1, 2020",2020,1
226790,Wilfred Ndidi,http://sofifa.com/player/226790/wilfred-ndidi/210005/,Right,CB,2,4,3,1,1,Prime,2017 - 2024,"Jan 5, 2017",2017,4
163587,Kasper Schmeichel,http://sofifa.com/player/163587/kasper-schmeichel/210005/,Right,GK,1,3,1,2,0,Veteran,2011 - 2023,"Jun 1, 2011",2011,10
239053,Federico Valverde,http://sofifa.com/player/239053/federico-valverde/210005/,Right,CM,1,3,3,1,0,Young,2016 - 2025,"Jul 22, 2016",2016,5


# DIMENSION PAIS

In [0]:
dim_nationality = (
    df_silver
    .select("nationality")
    .dropDuplicates()
    .withColumn("nationality_id", monotonically_increasing_id())
)

display(dim_nationality.limit(20))

dim_nationality.write.mode("overwrite").saveAsTable(
    "fifa_catalog.gold.dim_nationality"
)


nationality,nationality_id
Belgium,0
England,1
France,2
Algeria,3
Brazil,4
Nigeria,5
Denmark,6
Uruguay,7
Netherlands,8
Spain,9


# DIMENSION CLUBES

In [0]:
dim_club = (
    df_silver
    .select("team_name")
    .dropDuplicates()
    .withColumn("club_id", monotonically_increasing_id())
)

display(dim_club.limit(20))
dim_club.write.mode("overwrite").saveAsTable(
    "fifa_catalog.gold.dim_club"
)


team_name,club_id
Manchester City,0
Tottenham Hotspur,1
FC Barcelona,2
Chelsea,3
Juventus,4
Leicester City,5
Real Madrid,6
Manchester United,7
Everton,8
VfL Wolfsburg,9


# DIMENSION POSICIONES 

In [0]:
dim_position = (
    df_silver
    .select("best_position")
    .dropDuplicates()
    .withColumn("position_id", monotonically_increasing_id())
)
dim_position.write.mode("overwrite").saveAsTable(
    "fifa_catalog.gold.dim_position"
)
display(dim_position.limit(20))

best_position,position_id
CAM,0
ST,1
RW,2
CB,3
CM,4
GK,5
LB,6
CDM,7
LW,8
LM,9


# TABLA DE HECHOS (ESTADISTICAS)

In [0]:
fact_player_stats = (
    df_silver
    # joins para crear las FKs
    .join(dim_position, on="best_position", how="left")
    .join(dim_nationality, on="nationality", how="left")
    .join(dim_club, on="team_name", how="left")
    .select(
        df_silver.id.alias("player_id"),
        "position_id",
        "nationality_id",
        "club_id",
        "age",
        "overall_rating",
        "potential",
        "height",
        "weight",
        "value_player",
        "wage",
        "total_stats",
        "base_stats",
        "offensive_score",
        "defensive_score",
        "clutch_score"
    )
)
fact_player_stats.write.mode("overwrite").saveAsTable(
    "fifa_catalog.gold.fact_player_stats"
)
display(fact_player_stats.limit(20))

player_id,position_id,nationality_id,club_id,age,overall_rating,potential,height,weight,value_player,wage,total_stats,base_stats,offensive_score,defensive_score,clutch_score
192985,1,57,400,29,91,91,180.3,69.9,Premium,370000,2304,485,85.83,64.4,92.0
202126,14,44,401,26,88,89,188.0,88.9,Premium,220000,2172,449,81.67,48.0,88.0
194765,14,145,179,29,87,87,175.3,73.0,High,290000,2288,465,85.33,54.4,88.67
204485,6,84,400,29,85,85,177.8,67.1,High,210000,2034,430,85.17,37.0,83.0
184087,8,57,401,31,85,85,185.4,81.2,High,130000,2031,423,60.5,85.0,77.67
164240,8,133,40,35,85,85,182.9,78.9,Mid,93000,2037,420,61.67,85.2,79.0
230658,13,133,533,23,84,89,170.2,73.0,High,98000,2163,454,77.67,73.8,87.0
226790,8,106,402,23,84,88,182.9,73.9,High,96000,2120,446,66.67,85.8,78.0
163587,10,21,402,33,84,84,188.0,88.9,Mid,90000,1366,464,23.5,33.0,69.0
239053,13,107,0,21,83,90,182.9,78.0,High,135000,2173,474,77.33,78.6,82.0


## Fin del pipeline

Las 5 tablas Gold ya están disponibles en `fifa_catalog.gold` para consultas de explotación (top jugadores por rating, salario medio por club, distribución por posición, etc.).